## Work Sample

Gerald M. Thomas, MBA/MS
Data Scientist 

A big challenge for the VA is how to maintain strict compliance with the landmark 8-0 Supreme Court decision involving Kingdomware Technologies, Inc. v. United States, 579 U.S. 162 (2016). Implementation can be a significant challenge and this Supreme Court opinion led to rise of "Rent-a-Veteran" fraud schemes. The following is a proposal on how the VA can use commercially available large language models to research technical nuances of the landmark Supreme Court decision. 

Because this prototype results involves off-site Large Language Models, the Government must be careful not to use any confidential information, PII, or otherwise sensitive information in any of the prompts. LLMs are a great resource to quickly gather information from complex sources; however, they have a major security vulnerability since the data is stored off-site.

In [ ]:
import os
import json
import requests
from openai import OpenAI

# Initialize the OpenAI client (requires openai >= 1.0.0)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## Prompt Engineering

The hall mark of this code involves using prompt engineering to pass optimized legal prompts through an Open API. This model will return the optimized prompt and identify the right questions to ask a separate large language model trained on the Code of Federal Regulations, the U.S. Code, and relevant Supreme Court case law. This model does not replace legal analysis, but can be used by OIG and OGC stakeholders to quickly understand the legal rationale for their decisions. To use this code, the agency requires a FEDRAMP-approved API key with OpenAI.


In [ ]:
def generate_legal_synthesis_prompt(agency_context, target_fraud_vectors):
    """
    Constructs a custom master prompt by injecting local variables into a template.
    """
    prompt_template = """
    You are an expert federal contracting attorney and data scientist for the {agency_context}.
    Analyze the provided Supreme Court case text and extract the following:
    
    1. A 2-sentence summary of the ruling regarding the 'Rule of Two'.
    2. How this ruling alters the financial incentives for large, non-veteran corporations.
    3. Three specific Rent-a-Veteran fraud schemes (focusing on {target_fraud_vectors}) that are likely to increase as a result of this ruling.
    
    Output strictly in valid JSON format with the following keys:
    "case_name", "rule_of_two_summary", "financial_incentive_shift", "projected_fraud_vectors"
    """
    
    formatted_prompt = prompt_template.format(
        agency_context=agency_context,
        target_fraud_vectors=target_fraud_vectors
    )
    
    return formatted_prompt

def fetch_case_law(case_id):
    """
    Retrieves the raw legal text of a Supreme Court decision via the OpenCase API.
    """
    opencase_api_key = os.environ.get("OPENCASE_API_KEY")
    
    # Standard REST API call to ingest the court data
    url = f"https://api.opencase.com/v1/cases/{case_id}/text"
    headers = {
        "Authorization": f"Bearer {opencase_api_key}",
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        # Graceful fallback for the GitHub PoC if the OpenCase API key is missing or endpoint is down
        print(f"Warning: OpenCase API unavailable ({e}). Defaulting to Kingdomware fallback text.")
        return """
        Kingdomware Technologies, Inc. v. United States, 579 U.S. 162 (2016)
        Held: The Department of Veterans Affairs must use the "Rule of Two" when awarding contracts, 
        restricting competition to veteran-owned small businesses if at least two such businesses 
        are expected to submit reasonable offers. This mandate applies to all VA contracts, 
        regardless of whether the Department has already met its annual minimum contracting goals.
        """

## Legal Analysis

A common tool to perform complex legal analysis is OpenCase.com to access a commercially available specialty API. This API is already fine-tuned on legal materials, including the Code of Federal Regulations, the Appellate Circuit Courts, and the U.S. Code. However, the Government can pass the API into any commercially available LLM fined-tuned on this data, or even a custom OpenAI developer portal fined-tuned on a federal law library. To use, the Government will need a FEDRAMP-approved API key specifically fine-tuned on federal regulations, statutes passed by Congress, and federal case law.

In [ ]:






def run_legal_synthesis():
    # 1. Define local variables tailored to the VA OSDBU mission
    agency = "VA Office of Small and Disadvantaged Business Utilization (OSDBU)"
    vectors = "pass-throughs, figureheads, and shared infrastructure schemes"
    
    # 2. Ingest the actual case law via OpenCase API (Kingdomware Case ID: 14-916)
    print("Fetching Supreme Court data from OpenCase API...")
    case_text = fetch_case_law("14-916")
    
    # 3. Build the final system prompt string
    system_prompt = generate_legal_synthesis_prompt(
        agency_context=agency, 
        target_fraud_vectors=vectors
    )
    
    # 4. Pass the formatted prompt and ingested case text to the OpenAI API
    print("Processing legal synthesis via OpenAI API...")
    response = client.chat.completions.create(
        model="gpt-4o",
        response_format={ "type": "json_object" }, 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Case Text to analyze:\n\n{case_text}"}
        ],
        temperature=0.1 # Crucial for legal data: Low temperature prevents hallucination
    )
    
    return response.choices[0].message.content

# Example execution
if __name__ == "__main__":
    # Ensure environmental variables are loaded before execution
    if not os.environ.get("OPENAI_API_KEY"):
        print("Error: OPENAI_API_KEY environment variable is missing.")
    else:
        draft = run_legal_synthesis()
        
        # Parse and print the JSON cleanly for demonstration
        print("\n--- SYNTHESIS COMPLETE ---")
        parsed_json = json.loads(draft)
        print(json.dumps(parsed_json, indent=4))